# Notebook 04 — Heterogeneous Graph Construction

## Dynamic Heterogeneous Graph Neural Network for Bank Marketing Prediction using GraphSAGE

### Objective

Convert the verified Bank Marketing customer data into a meaningful **heterogeneous graph** using PyTorch Geometric `HeteroData`.

The graph design is:

```text
                         ┌── job
                         ├── education
                         ├── marital
                         ├── contact
Customer ────────────────┼── month
                         ├── housing
                         ├── loan
                         ├── default
                         └── poutcome
```

Each customer becomes a `customer` node.

Each categorical value becomes an entity node belonging to its corresponding node type.

Example:

```text
customer_42 ──works_as──> job_admin
customer_42 ──has_education──> education_secondary
customer_42 ──has_marital──> marital_married
```

### Important design principles

- Graph construction uses the **raw categorical identity** of each entity.
- The customer feature matrix comes from the verified Notebook 03 preprocessing pipeline.
- The target is stored only on customer nodes.
- Train/validation/test masks are preserved at the customer-node level.
- The test mask is not used for training.
- No model is trained in this notebook.
- No graph neural network is trained in this notebook.

### Why a heterogeneous graph?

The dataset contains multiple entity types and customer-to-entity relationships. A heterogeneous graph lets the model distinguish:

```text
customer → job
```

from:

```text
customer → education
```

rather than treating every relationship as the same type.

This preserves semantic information that a simple homogeneous graph would lose.


# 1. Required Inputs

This notebook expects the artifacts produced by Notebook 03.

```text
artifacts/
├── models/
│   └── preprocessor.joblib
└── metadata/
    └── feature_schema.json

data/
├── interim/
│   ├── train_engineered.csv
│   ├── validation_engineered.csv
│   └── test_engineered.csv
└── processed/
    ├── X_train.npy
    ├── X_validation.npy
    ├── X_test.npy
    ├── y_train.npy
    ├── y_validation.npy
    └── y_test.npy
```

The graph will use the complete customer population so that each customer has a node and can be assigned to exactly one of the train/validation/test masks.


In [ ]:
# ============================================================
# 1. Imports and Configuration
# ============================================================

from pathlib import Path
import json
import sys
import importlib.util

import numpy as np
import pandas as pd
import torch

from IPython.display import display

# Check PyTorch Geometric availability before doing graph work.
PYG_AVAILABLE = importlib.util.find_spec("torch_geometric") is not None

if not PYG_AVAILABLE:
    raise ImportError(
        "PyTorch Geometric is not installed. "
        "Install a compatible PyTorch Geometric version before running Notebook 04."
    )

from torch_geometric.data import HeteroData

RANDOM_STATE = 42
TARGET_COLUMN = "y"

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name.lower() == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

DATA_PATH = PROJECT_ROOT / "data" / "raw" / "bank-full.csv"

INTERIM_DIR = PROJECT_ROOT / "data" / "interim"
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"

MODELS_DIR = PROJECT_ROOT / "artifacts" / "models"
METADATA_DIR = PROJECT_ROOT / "artifacts" / "metadata"
GRAPH_DIR = PROJECT_ROOT / "artifacts" / "graph"

GRAPH_DIR.mkdir(parents=True, exist_ok=True)

GRAPH_PATH = GRAPH_DIR / "bank_heterodata.pt"
GRAPH_METADATA_PATH = GRAPH_DIR / "graph_metadata.json"

print("Project root:", PROJECT_ROOT)
print("Graph output:", GRAPH_PATH)
print("PyTorch version:", torch.__version__)


# 2. Verify Notebook 03 Artifacts

Do not construct a graph from fabricated or missing artifacts.

All required files must exist before proceeding.


In [ ]:
# ============================================================
# 2. Verify Required Inputs
# ============================================================

required_inputs = [
    MODELS_DIR / "preprocessor.joblib",
    METADATA_DIR / "feature_schema.json",

    INTERIM_DIR / "train_engineered.csv",
    INTERIM_DIR / "validation_engineered.csv",
    INTERIM_DIR / "test_engineered.csv",

    PROCESSED_DIR / "X_train.npy",
    PROCESSED_DIR / "X_validation.npy",
    PROCESSED_DIR / "X_test.npy",

    PROCESSED_DIR / "y_train.npy",
    PROCESSED_DIR / "y_validation.npy",
    PROCESSED_DIR / "y_test.npy",
]

missing_inputs = [
    str(path)
    for path in required_inputs
    if not path.exists()
]

if missing_inputs:
    raise FileNotFoundError(
        "Required Notebook 03 artifacts are missing:\n"
        + "\n".join(missing_inputs)
    )

print("All required Notebook 03 artifacts are present.")


# 3. Load Raw Dataset and Notebook 03 Outputs

The raw categorical values are required to create entity nodes.

The engineered split files preserve the original customer index, which lets us reconstruct the global train/validation/test masks.


In [ ]:
# ============================================================
# 3. Load Data and Artifacts
# ============================================================

df = pd.read_csv(DATA_PATH, sep=";")

train_df = pd.read_csv(INTERIM_DIR / "train_engineered.csv", index_col=0)
val_df = pd.read_csv(INTERIM_DIR / "validation_engineered.csv", index_col=0)
test_df = pd.read_csv(INTERIM_DIR / "test_engineered.csv", index_col=0)

X_train = np.load(PROCESSED_DIR / "X_train.npy")
X_val = np.load(PROCESSED_DIR / "X_validation.npy")
X_test = np.load(PROCESSED_DIR / "X_test.npy")

y_train = np.load(PROCESSED_DIR / "y_train.npy")
y_val = np.load(PROCESSED_DIR / "y_validation.npy")
y_test = np.load(PROCESSED_DIR / "y_test.npy")

with open(METADATA_DIR / "feature_schema.json", "r", encoding="utf-8") as f:
    feature_schema = json.load(f)

print("Raw dataset:", df.shape)
print("Train engineered:", train_df.shape)
print("Validation engineered:", val_df.shape)
print("Test engineered:", test_df.shape)

print("\nProcessed matrices:")
print("X_train:", X_train.shape)
print("X_val  :", X_val.shape)
print("X_test :", X_test.shape)


# 4. Verify Customer Index Alignment

The graph requires one customer node per original customer row.

The customer-node order is defined as the original raw dataset row order.

This gives us a deterministic mapping:

```text
raw dataframe index
        ↓
customer node ID
```

The train/validation/test engineered files must contain mutually exclusive original indices.


In [ ]:
# ============================================================
# 4. Verify Global Customer Index Alignment
# ============================================================

raw_indices = df.index.to_numpy()

train_indices = train_df.index.to_numpy()
val_indices = val_df.index.to_numpy()
test_indices = test_df.index.to_numpy()

assert len(raw_indices) == len(df)

assert set(train_indices).isdisjoint(set(val_indices))
assert set(train_indices).isdisjoint(set(test_indices))
assert set(val_indices).isdisjoint(set(test_indices))

assert set(train_indices) | set(val_indices) | set(test_indices) == set(raw_indices)

assert len(train_indices) == len(y_train)
assert len(val_indices) == len(y_val)
assert len(test_indices) == len(y_test)

print("Customer index alignment verified.")
print("Total customers:", len(raw_indices))
print("Train customers:", len(train_indices))
print("Validation customers:", len(val_indices))
print("Test customers:", len(test_indices))


# 5. Define Candidate Graph Entities

The EDA identified nine candidate categorical entities:

```text
job
education
marital
contact
month
housing
loan
default
poutcome
```

Each becomes a separate node type.

We intentionally avoid creating nodes for every numerical value because that would create unnecessary graph complexity and would not represent a natural categorical entity.


In [ ]:
# ============================================================
# 5. Define Graph Entities
# ============================================================

candidate_graph_entities = [
    "job",
    "education",
    "marital",
    "contact",
    "month",
    "housing",
    "loan",
    "default",
    "poutcome",
]

missing_entities = [
    column
    for column in candidate_graph_entities
    if column not in df.columns
]

if missing_entities:
    raise KeyError(
        "Candidate graph entities missing from raw dataset: "
        + ", ".join(missing_entities)
    )

print("Graph entity types:")
for entity in candidate_graph_entities:
    print(f"- {entity}: {df[entity].nunique()} unique values")


# 6. Build Global Train / Validation / Test Masks

Masks live on the `customer` node type.

```text
customer.train_mask
customer.val_mask
customer.test_mask
```

Only one mask is true for each customer.

These masks will later be consumed by Notebook 05 during GraphSAGE training.


In [ ]:
# ============================================================
# 6. Build Customer Masks
# ============================================================

num_customers = len(df)

train_mask = torch.zeros(num_customers, dtype=torch.bool)
val_mask = torch.zeros(num_customers, dtype=torch.bool)
test_mask = torch.zeros(num_customers, dtype=torch.bool)

train_mask[train_indices] = True
val_mask[val_indices] = True
test_mask[test_indices] = True

assert torch.all(
    train_mask.to(torch.int8)
    + val_mask.to(torch.int8)
    + test_mask.to(torch.int8)
    == 1
)

print("Customer masks created.")
print("Train:", int(train_mask.sum()))
print("Validation:", int(val_mask.sum()))
print("Test:", int(test_mask.sum()))


# 7. Create Customer Node Features

The customer node features are the exact transformed features generated by Notebook 03.

The processed matrices are concatenated back into the original customer order using the preserved indices.

This is important:

```text
Notebook 03 preprocessing
          ↓
processed customer features
          ↓
global customer order
          ↓
customer.x
```

No new scaling or encoding is performed here.


In [ ]:
# ============================================================
# 7. Reconstruct Global Customer Feature Matrix
# ============================================================

feature_dimension = X_train.shape[1]

X_customer = np.empty(
    (num_customers, feature_dimension),
    dtype=np.float32
)

X_customer[train_indices] = X_train.astype(np.float32)
X_customer[val_indices] = X_val.astype(np.float32)
X_customer[test_indices] = X_test.astype(np.float32)

assert X_customer.shape == (
    num_customers,
    feature_dimension
)

assert np.isfinite(X_customer).all()

customer_x = torch.from_numpy(X_customer)

print("Customer feature matrix:")
print("Shape:", customer_x.shape)
print("dtype:", customer_x.dtype)


# 8. Encode the Target

The target was established during Notebook 03 as:

```text
no  → 0
yes → 1
```

The graph stores the target on customer nodes only.

Entity nodes do not receive target labels.


In [ ]:
# ============================================================
# 8. Build Customer Target Tensor
# ============================================================

target_mapping = {
    "no": 0,
    "yes": 1,
}

if set(df[TARGET_COLUMN].unique()) - set(target_mapping):
    raise ValueError(
        f"Unexpected target values: {df[TARGET_COLUMN].unique().tolist()}"
    )

y_global = (
    df[TARGET_COLUMN]
    .map(target_mapping)
    .to_numpy(dtype=np.int64)
)

customer_y = torch.from_numpy(y_global)

assert customer_y.shape[0] == num_customers
assert set(customer_y.unique().tolist()).issubset({0, 1})

print("Target tensor:")
print("Shape:", customer_y.shape)
print("Class counts:")
print(
    pd.Series(y_global)
    .value_counts()
    .sort_index()
    .rename(index={0: "no", 1: "yes"})
)


# 9. Build Entity Node Dictionaries

For each categorical entity:

1. Obtain unique category values from the raw dataset.
2. Sort them deterministically.
3. Assign an integer node ID.
4. Store the mapping for reproducibility.

Example:

```text
job
  0 → admin.
  1 → blue-collar
  2 → entrepreneur
  ...
```

The mappings are saved in graph metadata.


In [ ]:
# ============================================================
# 9. Build Entity Mappings
# ============================================================

entity_mappings = {}
entity_node_counts = {}

for entity in candidate_graph_entities:

    values = (
        df[entity]
        .astype(str)
        .drop_duplicates()
        .sort_values()
        .tolist()
    )

    value_to_id = {
        value: idx
        for idx, value in enumerate(values)
    }

    entity_mappings[entity] = value_to_id
    entity_node_counts[entity] = len(values)

print("Entity node counts:")
for entity, count in entity_node_counts.items():
    print(f"{entity:12s}: {count}")


# 10. Construct Customer → Entity Edges

For each entity type, create one edge for every customer.

Example:

```text
customer 0 ──customer_has_job──> job 3
customer 1 ──customer_has_job──> job 1
customer 2 ──customer_has_job──> job 5
```

Every customer receives exactly one outgoing relation for each categorical entity because each raw categorical field contains one value per customer.

The reverse relation is also created so message passing can flow in both directions.


In [ ]:
# ============================================================
# 10. Build Edge Indices
# ============================================================

edge_indices = {}
edge_counts = {}

for entity in candidate_graph_entities:

    value_to_id = entity_mappings[entity]

    entity_ids = (
        df[entity]
        .astype(str)
        .map(value_to_id)
        .to_numpy(dtype=np.int64)
    )

    customer_ids = np.arange(num_customers, dtype=np.int64)

    forward_edge_index = torch.from_numpy(
        np.vstack([
            customer_ids,
            entity_ids
        ])
    ).long()

    reverse_edge_index = torch.from_numpy(
        np.vstack([
            entity_ids,
            customer_ids
        ])
    ).long()

    edge_indices[
        ("customer", f"has_{entity}", entity)
    ] = forward_edge_index

    edge_indices[
        (entity, f"rev_has_{entity}", "customer")
    ] = reverse_edge_index

    edge_counts[
        f"customer__has_{entity}__{entity}"
    ] = int(forward_edge_index.shape[1])

    edge_counts[
        f"{entity}__rev_has_{entity}__customer"
    ] = int(reverse_edge_index.shape[1])

print("Edge construction complete.")

for edge_type, edge_index in edge_indices.items():
    print(f"{edge_type}: {edge_index.shape}")


# 11. Create PyTorch Geometric HeteroData

The final graph contains:

### Node types

```text
customer
job
education
marital
contact
month
housing
loan
default
poutcome
```

### Edge types

For each entity:

```text
customer → entity
entity → customer
```

This creates a semantic heterogeneous graph with explicit relation types.


In [ ]:
# ============================================================
# 11. Build HeteroData
# ============================================================

data = HeteroData()

# Customer node
data["customer"].x = customer_x
data["customer"].y = customer_y

data["customer"].train_mask = train_mask
data["customer"].val_mask = val_mask
data["customer"].test_mask = test_mask

# Entity nodes
for entity in candidate_graph_entities:

    num_nodes = entity_node_counts[entity]

    # Entity nodes do not need learned features yet.
    # A simple one-dimensional constant feature is used as an
    # explicit placeholder. Notebook 05 can learn entity embeddings.
    data[entity].x = torch.ones(
        (num_nodes, 1),
        dtype=torch.float32
    )

# Edges
for edge_type, edge_index in edge_indices.items():
    data[edge_type].edge_index = edge_index

print(data)


# 12. Graph Statistics

Generate explicit statistics for:

- Node counts
- Edge counts
- Node feature dimensions
- Train/validation/test customer counts
- Number of node types
- Number of edge types

These statistics are saved for later model development.


In [ ]:
# ============================================================
# 12. Graph Statistics
# ============================================================

node_statistics = {}

for node_type in data.node_types:
    node_store = data[node_type]

    node_statistics[node_type] = {
        "num_nodes": int(node_store.num_nodes),
        "feature_dimension": (
            int(node_store.x.shape[1])
            if getattr(node_store, "x", None) is not None
            else None
        )
    }

edge_statistics = {}

for edge_type in data.edge_types:
    edge_statistics["__".join(edge_type)] = {
        "num_edges": int(data[edge_type].edge_index.shape[1]),
        "source_type": edge_type[0],
        "relation": edge_type[1],
        "destination_type": edge_type[2],
    }

print("Node statistics:")
display(pd.DataFrame(node_statistics).T)

print("\nEdge statistics:")
display(pd.DataFrame(edge_statistics).T)


# 13. Validate Graph Structure

The following invariants must hold:

1. Every customer has one node.
2. Every customer has one relation to every entity type.
3. Reverse relations have the same number of edges.
4. Customer masks cover every customer exactly once.
5. Customer features and targets have matching lengths.
6. Entity node IDs are within valid ranges.


In [ ]:
# ============================================================
# 13. Graph Integrity Checks
# ============================================================

assert data["customer"].num_nodes == num_customers
assert data["customer"].x.shape[0] == num_customers
assert data["customer"].y.shape[0] == num_customers

assert data["customer"].train_mask.sum() +        data["customer"].val_mask.sum() +        data["customer"].test_mask.sum() == num_customers

for entity in candidate_graph_entities:

    forward_type = ("customer", f"has_{entity}", entity)
    reverse_type = (entity, f"rev_has_{entity}", "customer")

    forward_edges = data[forward_type].edge_index
    reverse_edges = data[reverse_type].edge_index

    assert forward_edges.shape[1] == num_customers
    assert reverse_edges.shape[1] == num_customers

    # Source customer IDs are valid.
    assert forward_edges[0].min() >= 0
    assert forward_edges[0].max() < num_customers

    # Entity IDs are valid.
    assert forward_edges[1].min() >= 0
    assert forward_edges[1].max() < entity_node_counts[entity]

    # Reverse relation points to the same valid node spaces.
    assert reverse_edges[0].min() >= 0
    assert reverse_edges[0].max() < entity_node_counts[entity]

    assert reverse_edges[1].min() >= 0
    assert reverse_edges[1].max() < num_customers

print("All graph integrity checks passed.")


# 14. Validate One-to-One Customer Entity Relationships

Each customer has exactly one value for every categorical entity.

Therefore, each forward customer → entity relation should contain exactly one edge for every customer ID.

This confirms that the graph construction matches the tabular semantics.


In [ ]:
# ============================================================
# 14. Validate Customer Degree
# ============================================================

degree_checks = []

for entity in candidate_graph_entities:

    edge_index = data[
        ("customer", f"has_{entity}", entity)
    ].edge_index

    customer_degree = torch.bincount(
        edge_index[0],
        minlength=num_customers
    )

    degree_checks.append({
        "entity": entity,
        "min_customer_degree": int(customer_degree.min()),
        "max_customer_degree": int(customer_degree.max()),
        "all_customers_exactly_one_edge": bool(
            torch.all(customer_degree == 1)
        )
    })

degree_check_df = pd.DataFrame(degree_checks)
display(degree_check_df)

assert degree_check_df["all_customers_exactly_one_edge"].all()

print("Every customer has exactly one edge to every entity type.")


# 15. Check Graph Connectivity

The graph is bipartite at each individual relation:

```text
customer ↔ entity
```

Because customers can share the same entity nodes, the graph creates natural multi-hop relationships.

For example:

```text
customer A
    ↓
job = technician
    ↑
customer B
```

This means GraphSAGE can aggregate information from structurally related customers through shared entity nodes.


In [ ]:
# ============================================================
# 15. Connectivity-Oriented Statistics
# ============================================================

connectivity_summary = []

for entity in candidate_graph_entities:

    entity_values = df[entity].astype(str)

    counts = entity_values.value_counts()

    connectivity_summary.append({
        "entity": entity,
        "unique_entity_nodes": int(len(counts)),
        "largest_shared_group": int(counts.max()),
        "smallest_group": int(counts.min()),
        "average_customers_per_entity": float(counts.mean()),
    })

connectivity_df = pd.DataFrame(connectivity_summary)

display(connectivity_df)


# 16. Save Graph

Save the complete `HeteroData` object:

```text
artifacts/graph/bank_heterodata.pt
```

This file contains:

- customer features
- customer targets
- train/validation/test masks
- entity nodes
- heterogeneous edge indices

No model weights are stored in this file.


In [ ]:
# ============================================================
# 16. Save Heterogeneous Graph
# ============================================================

torch.save(data, GRAPH_PATH)

assert GRAPH_PATH.exists()

print(f"Graph saved to: {GRAPH_PATH}")
print(f"File size: {GRAPH_PATH.stat().st_size / (1024 ** 2):.2f} MB")


# 17. Save Graph Metadata

Metadata records the exact graph design and mappings so the graph can be interpreted and reproduced later.


In [ ]:
# ============================================================
# 17. Save Graph Metadata
# ============================================================

graph_metadata = {
    "graph_type": "heterogeneous_customer_entity_graph",

    "node_types": list(data.node_types),

    "edge_types": [
        list(edge_type)
        for edge_type in data.edge_types
    ],

    "candidate_graph_entities": candidate_graph_entities,

    "node_statistics": node_statistics,

    "edge_statistics": edge_statistics,

    "connectivity_statistics": connectivity_df.to_dict(
        orient="records"
    ),

    "customer_nodes": {
        "count": num_customers,
        "feature_dimension": feature_dimension,
        "train_count": int(train_mask.sum()),
        "validation_count": int(val_mask.sum()),
        "test_count": int(test_mask.sum()),
    },

    "entity_mappings": entity_mappings,

    "target": {
        "column": TARGET_COLUMN,
        "mapping": target_mapping,
    },

    "feature_source": {
        "preprocessing_artifact": str(
            (MODELS_DIR / "preprocessor.joblib").relative_to(PROJECT_ROOT)
        ),
        "feature_schema": str(
            (METADATA_DIR / "feature_schema.json").relative_to(PROJECT_ROOT)
        ),
    },

    "design_notes": [
        "Customer nodes contain the exact transformed features from Notebook 03.",
        "Entity nodes represent categorical values.",
        "Each customer has one edge to each candidate entity type.",
        "Reverse relations are included for bidirectional message passing.",
        "Target labels are stored only on customer nodes.",
        "Train/validation/test masks are stored only on customer nodes.",
        "No model training occurs in Notebook 04."
    ]
}

with open(GRAPH_METADATA_PATH, "w", encoding="utf-8") as f:
    json.dump(graph_metadata, f, indent=2)

assert GRAPH_METADATA_PATH.exists()

print(f"Graph metadata saved to: {GRAPH_METADATA_PATH}")


# 18. Reload Graph and Verify Serialization

The saved graph must be loadable before Notebook 04 is considered complete.

PyTorch 2.6+ may default to a restricted `weights_only=True` loading mode. Because this artifact intentionally contains a PyTorch Geometric `HeteroData` object, we explicitly use `weights_only=False` when supported.


In [ ]:
# ============================================================
# 18. Reload Saved Graph
# ============================================================

try:
    loaded_data = torch.load(
        GRAPH_PATH,
        map_location="cpu",
        weights_only=False
    )
except TypeError:
    # Compatibility with older PyTorch versions.
    loaded_data = torch.load(
        GRAPH_PATH,
        map_location="cpu"
    )

assert isinstance(loaded_data, HeteroData)

print("Saved HeteroData reloaded successfully.")
print(loaded_data)


In [ ]:
# ============================================================
# 19. Compare Original and Reloaded Graph
# ============================================================

assert loaded_data.node_types == data.node_types
assert loaded_data.edge_types == data.edge_types

for node_type in data.node_types:
    assert loaded_data[node_type].num_nodes == data[node_type].num_nodes

    if hasattr(data[node_type], "x"):
        assert torch.equal(
            loaded_data[node_type].x,
            data[node_type].x
        )

for edge_type in data.edge_types:
    assert torch.equal(
        loaded_data[edge_type].edge_index,
        data[edge_type].edge_index
    )

assert torch.equal(
    loaded_data["customer"].y,
    data["customer"].y
)

assert torch.equal(
    loaded_data["customer"].train_mask,
    data["customer"].train_mask
)

assert torch.equal(
    loaded_data["customer"].val_mask,
    data["customer"].val_mask
)

assert torch.equal(
    loaded_data["customer"].test_mask,
    data["customer"].test_mask
)

print("Reloaded graph matches the original graph.")


# 20. Final Graph Summary

The following summary is the handoff from Notebook 04 to Notebook 05.

```text
Customer features
        +
Customer ↔ Entity relations
        +
Train / Validation / Test masks
        ↓
Heterogeneous HeteroData
        ↓
GraphSAGE training
```

Notebook 05 will be responsible for model architecture, loss, optimization, validation metrics, checkpointing, and training curves.


In [ ]:
# ============================================================
# 20. Final Summary
# ============================================================

print("=" * 85)
print("HETEROGENEOUS GRAPH SUMMARY")
print("=" * 85)

print("Node types:")
for node_type in data.node_types:
    print(
        f"  {node_type:12s} | "
        f"nodes={data[node_type].num_nodes:,} | "
        f"features={data[node_type].x.shape[1]}"
    )

print("\nEdge types:")
for edge_type in data.edge_types:
    print(
        f"  {edge_type[0]:12s} "
        f"--{edge_type[1]:18s}--> "
        f"{edge_type[2]:12s} | "
        f"edges={data[edge_type].edge_index.shape[1]:,}"
    )

print("\nCustomer masks:")
print(f"  train      : {int(train_mask.sum()):,}")
print(f"  validation : {int(val_mask.sum()):,}")
print(f"  test       : {int(test_mask.sum()):,}")

print("\nGraph artifact:")
print(f"  {GRAPH_PATH}")

print("\nMetadata:")
print(f"  {GRAPH_METADATA_PATH}")


# 21. Final Verification Checklist

Notebook 04 is successful only when:

- [x] Notebook 03 artifacts exist.
- [x] Raw customer ordering is preserved.
- [x] Customer feature matrix is reconstructed correctly.
- [x] Target mapping is verified.
- [x] Train/validation/test masks are mutually exclusive and exhaustive.
- [x] Nine candidate entity types are represented.
- [x] Customer → entity edges exist.
- [x] Reverse entity → customer edges exist.
- [x] Every customer has exactly one relation to every entity type.
- [x] Entity IDs are valid.
- [x] Graph node and edge statistics are generated.
- [x] `HeteroData` is serialized.
- [x] Graph metadata is saved.
- [x] Saved graph can be reloaded.
- [x] Reloaded graph matches the original graph.

## Required Artifacts

```text
artifacts/
└── graph/
    ├── bank_heterodata.pt
    └── graph_metadata.json
```

### Important

No GraphSAGE model is trained here.

No model performance is reported here.

The next stage is **Notebook 05 — GraphSAGE Training**.


In [ ]:
# ============================================================
# 21. FINAL AUTOMATED VERIFICATION
# ============================================================

assert isinstance(data, HeteroData)
assert GRAPH_PATH.exists()
assert GRAPH_METADATA_PATH.exists()

assert data["customer"].num_nodes == 45211
assert data["customer"].x.shape[0] == 45211
assert data["customer"].y.shape[0] == 45211

assert int(data["customer"].train_mask.sum()) == len(train_indices)
assert int(data["customer"].val_mask.sum()) == len(val_indices)
assert int(data["customer"].test_mask.sum()) == len(test_indices)

assert torch.all(
    data["customer"].train_mask.to(torch.int8)
    + data["customer"].val_mask.to(torch.int8)
    + data["customer"].test_mask.to(torch.int8)
    == 1
)

expected_node_types = [
    "customer",
    "job",
    "education",
    "marital",
    "contact",
    "month",
    "housing",
    "loan",
    "default",
    "poutcome",
]

assert set(data.node_types) == set(expected_node_types)

for entity in candidate_graph_entities:
    forward_type = ("customer", f"has_{entity}", entity)
    reverse_type = (entity, f"rev_has_{entity}", "customer")

    assert forward_type in data.edge_types
    assert reverse_type in data.edge_types

    assert data[forward_type].edge_index.shape[1] == 45211
    assert data[reverse_type].edge_index.shape[1] == 45211

print("=" * 85)
print("NOTEBOOK 04 VERIFICATION PASSED")
print("=" * 85)
print("Graph type       : Heterogeneous HeteroData")
print("Customer nodes   : 45,211")
print("Entity types     : 9")
print("Total node types : 10")
print("Total edge types :", len(data.edge_types))
print("Graph artifact   :", GRAPH_PATH)
print("Graph metadata   :", GRAPH_METADATA_PATH)
print("=" * 85)
print("Notebook 04 complete. Review and verify before proceeding to NEXT.")
